In [19]:
# ── Cell: Imports & project path setup ────────────────────────────────────
import sys, os, json, re
import ollama

# Make sure src/ is importable (notebook lives inside Main/)
sys.path.insert(0, os.path.abspath('.'))

from src import ActionConfig, SimulationConfig, OpenNTNPhyExecutor
print('Imports OK')

Imports OK


In [20]:
sim_cfg = SimulationConfig(
    scenario='sur',
    channel_model_name='SubUrban',
    perfect_csi=True,
    doppler_enabled=True,
    elevation_angle=50.0,
    carrier_frequency=2.0e9,

    batch_size=256,
    num_ut=1,
    num_ut_ant=1,
    num_bs_ant=1,

    #  FIXED (smaller size)
    num_ofdm_symbols=10,
    fft_size=128,

    subcarrier_spacing=15e3,
    cyclic_prefix_length=16,
    pilot_ofdm_symbol_indices=[2, 8],
)

print('SimulationConfig ready')

SimulationConfig ready


In [21]:
SYSTEM_PROMPT = """\
You are a 6G NTN physical-layer agent.

Output ONLY a JSON object with exactly:
{
  "modulation": "QPSK" or "16QAM",
  "code_rate": one of [0.33, 0.5, 0.67, 0.75],
  "power_boost_db": one of [-3.0, 0.0, 3.0, 6.0]
}

Rules:
- high reliability → QPSK, low code_rate, higher power
- high throughput → 16QAM, high code_rate
- energy saving → QPSK, moderate code_rate, low power
- low SNR → conservative
- high SNR → aggressive

Return ONLY JSON. No explanation.
""".strip()


ALLOWED_MODULATIONS  = ["QPSK", "16QAM"]
ALLOWED_CODE_RATES   = [0.33, 0.5, 0.67, 0.75]
ALLOWED_POWER_BOOSTS = [-3.0, 0.0, 3.0, 6.0]


def _snap(value, allowed):
    return min(allowed, key=lambda v: abs(v - value))


def _fallback(intent):
    if 'reliability' in intent.lower():
        return ActionConfig('QPSK', 0.33, 3.0)
    if 'throughput' in intent.lower():
        return ActionConfig('16QAM', 0.75, 0.0)
    return ActionConfig('QPSK', 0.5, 0.0)


def query_llm(snr_db, channel, interference, intent, model='llama3'):

    user_msg = (
        f"CSI: SNR={snr_db:.1f} dB, {channel}, {interference} interference.\n"
        f"Intent: {intent}."
    )

    try:
        resp = ollama.chat(
            model=model,
            messages=[
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user',   'content': user_msg},
            ],
            options={'temperature': 0.1},
        )
        raw = resp['message']['content'].strip()
        print("RAW LLM OUTPUT:", raw)

    except Exception as e:
        print(f'[LLM ERROR] {e} → fallback')
        return _fallback(intent)

    # 🔥 Robust JSON extraction
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if not m:
        print('[PARSE ERROR] → fallback')
        return _fallback(intent)

    try:
        cfg = json.loads(m.group(0))
    except:
        print('[JSON ERROR] → fallback')
        return _fallback(intent)

    # Validate
    mod = cfg.get('modulation', 'QPSK')
    if mod not in ALLOWED_MODULATIONS:
        mod = 'QPSK'

    cr = float(cfg.get('code_rate', 0.5))
    if cr not in ALLOWED_CODE_RATES:
        cr = _snap(cr, ALLOWED_CODE_RATES)

    pb = float(cfg.get('power_boost_db', 0.0))
    if pb not in ALLOWED_POWER_BOOSTS:
        pb = _snap(pb, ALLOWED_POWER_BOOSTS)

    return ActionConfig(modulation=mod, code_rate=cr, power_boost_db=pb)


print('LLM agent ready')

LLM agent ready


In [26]:
SCENARIOS = [
    (0.250,  'fast fading',     'high',   'high reliability'),
    (10.0, 'moderate fading', 'medium', 'high reliability'),
    (15.0, 'slow fading',     'low',    'high throughput'),
    (10.0, 'moderate fading', 'medium', 'energy saving'),
]

results = []

for snr_db, channel, interference, intent in SCENARIOS:

    print('\n' + '-'*60)
    print(f'SNR={snr_db} dB | {channel} | interference={interference}')
    print(f'Intent : {intent}')

    # LLM decision
    action_cfg = query_llm(snr_db, channel, interference, intent)

    print(f'Config : modulation={action_cfg.modulation}, '
          f'code_rate={action_cfg.code_rate}, ')
          #f'power_boost_db={action_cfg.power_boost_db} dB')

    # Simulation
    executor = OpenNTNPhyExecutor(action_cfg=action_cfg, sim_cfg=sim_cfg)
    metrics  = executor(float(snr_db))

    print(f'Result : BER={metrics["ber"]:.4e} | '
          f'BLER={metrics["bler"]:.4e} | '
          f'iSE={metrics["iSE"]:.4f} bpcu')

    results.append({
        'intent': intent,
        'snr_db': snr_db,
        'modulation': action_cfg.modulation,
        'code_rate': action_cfg.code_rate,
        #'power_boost_db': action_cfg.power_boost_db,
        **metrics,
    })


------------------------------------------------------------
SNR=0.25 dB | fast fading | interference=high
Intent : high reliability
RAW LLM OUTPUT: {
  "modulation": "QPSK",
  "code_rate": 0.33,
  "power_boost_db": -3.0
}
Config : modulation=QPSK, code_rate=0.33, 
Result : BER=3.3477e-01 | BLER=1.0000e+00 | iSE=0.0000 bpcu

------------------------------------------------------------
SNR=10.0 dB | moderate fading | interference=medium
Intent : high reliability
RAW LLM OUTPUT: {
  "modulation": "QPSK",
  "code_rate": 0.5,
  "power_boost_db": 3.0
}
Config : modulation=QPSK, code_rate=0.5, 
Result : BER=0.0000e+00 | BLER=0.0000e+00 | iSE=1.0000 bpcu

------------------------------------------------------------
SNR=15.0 dB | slow fading | interference=low
Intent : high throughput
RAW LLM OUTPUT: {
  "modulation": "16QAM",
  "code_rate": 0.75,
  "power_boost_db": 6.0
}
Config : modulation=16QAM, code_rate=0.75, 
Result : BER=1.7341e-01 | BLER=1.0000e+00 | iSE=0.0000 bpcu

----------------

In [23]:
print('\n' + '='*70)
print(f'{"Intent":<20} {"Mod":<7} {"Rate":<6} {"PwrDB":<7} {"BER":>10}  {"iSE":>8}')
print(f'{"-"*20} {"-"*7} {"-"*6} {"-"*7} {"-"*10}  {"-"*8}')

for r in results:
    print(f'{r["intent"]:<20} {r["modulation"]:<7} {r["code_rate"]:<6} '
          f'{r["power_boost_db"]:<7} {r["ber"]:>10.3e}  {r["iSE"]:>8.4f}')

print('='*70)


Intent               Mod     Rate   PwrDB          BER       iSE
-------------------- ------- ------ ------- ----------  --------
high reliability     QPSK    0.33   6.0      0.000e+00    0.6600
high reliability     QPSK    0.5    3.0      0.000e+00    1.0000
high throughput      16QAM   0.75   6.0      1.738e-01    0.0000
energy saving        QPSK    0.5    -3.0     0.000e+00    1.0000
